In [1]:
# Import required modules
import numpy as np
import os
import pandas as pd
from skimage import io
from feature_extraction.measure_haralick import measure_haralick_features


In [2]:
from dask.distributed import Client, LocalCluster

In [3]:

cluster = LocalCluster(n_workers=4)
client = Client(cluster)
print(client)

<Client: 'tcp://127.0.0.1:39073' processes=4 threads=24, memory=30.28 GiB>


In [4]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 24,Total memory: 30.28 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39073,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:43011,Total threads: 6
Dashboard: http://127.0.0.1:45661/status,Memory: 7.57 GiB
Nanny: tcp://127.0.0.1:44875,


In [5]:
# Indicate the path to the input image
input_image___path = os.path.join(os.getcwd(),"secondary_output")
input_image___path = os.path.join("/home/kai/Downloads")

# get the name of the input image
input_image___name = "260202_fov_example.ome.tif"

# open input image
input_real_image = io.imread(os.path.join(input_image___path, input_image___name))
print(input_real_image.shape)

# the first image of the channel axis is the segmentation mask. The following images are channels, to be analysed


(2720, 2720, 7)


the file is a 2720x2720 field of view with 7 channels.

The first of the 7 channels is the segmentation mask.

Objects are individual cells.

The remaining 6 channels are different imaged structures / imaging modalities.

In [6]:
%pdb

Automatic pdb calling has been turned ON


In [7]:
real_data_haralick_features = measure_haralick_features(image=input_real_image[...,1:3],
                                             label_image=input_real_image[...,0],
                                             props=['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM'],
                                            #props=['contrast',],# 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM'],
                                             distances=[5, 15, 49],
                                             angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                                             channel_axis=-1,
                                             window_shape=50,
                                             glcm_daskbag_kwargs={'npartitions': 4})

real_data_haralick_features

# single channel,, window_shape==50, 6 prop, 3 distance [5, 15, 49], 4 angles [0, np.pi/4, np.pi/2, 3*np.pi/4], 10 partitions -> 8m 16s
# double channels,, window_shape==50, 6 prop, 3 distance [5, 15, 49], 4 angles [0, np.pi/4, np.pi/2, 3*np.pi/4], 12 partitions -> 15m 33s



Default: Rescaling image to 8 intensity levels - indicate levels in glcm_graycomtx_kwargs to avoid this


/home/kai/projects/ACID/feature_extraction/measure_haralick.py:802: FutureWarning: `RegionProperties.intensity_image` is deprecated starting in version 0.26 and will be removed in version 2.0. Use `RegionProperties.image_intensity` instead. 
  sub_image = region.intensity_image[..., channel_index]  # pick channel


KeyboardInterrupt: 

> /home/kai/projects/ACID/feature_extraction/measure_haralick.py(614)graycoprops()
    612         weights = ( (I - J) ** 2).reshape((num_level, num_level, npone, npone))
    613     elif prop == 'dissimilarity':
--> 614         weights =( np.abs(I - J)).reshape((num_level, num_level, npone, npone))
    615     elif prop == 'homogeneity':
    616         weights =( 1.0 / (1.0 + (I - J) ** 2)).reshape((num_level, num_level, npone, npone))



Process Dask Worker process (from Nanny):
2026-02-17 10:53:57,972 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
  File "/home/kai/software/miniforge3/envs/acid_develop/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kai/software/miniforge3/envs/acid_develop/lib/python3.12/asyncio/base_events.py", line 691, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "/home/kai/software/miniforge3/envs/acid_develop/lib/python3.12/site-packages/distributed/nanny.py", line 985, in run
    await worker.finished()
  File "/home/kai/software/miniforge3/envs/acid_develop/lib/python3.12/site-packages/distributed/core.py", line 494, in finished
    await self._event_finished.wait()
  Fil

ipdb>  ll


    510 def graycoprops(P: np.ndarray, prop='contrast'):
    511     """Calculate texture properties of a GLCM.
    512 
    513     Compute a feature of a gray level co-occurrence matrix to serve as
    514     a compact summary of the matrix. The properties are computed as
    515     follows:
    516 
    517     - 'contrast': :math:`\\sum_{i,j=0}^{levels-1} P_{i,j}(i-j)^2`
    518     - 'dissimilarity': :math:`\\sum_{i,j=0}^{levels-1}P_{i,j}|i-j|`
    519     - 'homogeneity': :math:`\\sum_{i,j=0}^{levels-1}\\frac{P_{i,j}}{1+(i-j)^2}`
    520     - 'ASM': :math:`\\sum_{i,j=0}^{levels-1} P_{i,j}^2`
    521     - 'energy': :math:`\\sqrt{ASM}`
    522     - 'correlation':
    523         .. math:: \\sum_{i,j=0}^{levels-1} P_{i,j}\\left[\\frac{(i-\\mu_i) \\
    524                   (j-\\mu_j)}{\\sqrt{(\\sigma_i^2)(\\sigma_j^2)}}\\right]
    525     - 'mean': :math:`\\sum_{i=0}^{levels-1} i*P_{i}`
    526     - 'variance': :math:`\\sum_{i=0}^{levels-1} P_{i}*(i-mean)^2`
    527     - 's

In [ ]:
1
